## Import packages

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels
import plotly.io as pio
pio.renderers.default = "notebook"

In [3]:
import os
os.getcwd()

'/Users/yasmine/Documents/GitHub/class_datascience/2026/03_EDA-Visualization'

## Loda data

In [4]:
df = pd.read_csv("../02_Data-Cleaning/data_generated/data_intro_co2.csv")
df.head(15)

,Country,Year,CO2_emissions[MtCO2],population[milpeople],CO2_emissions_per_capita[tCO2/person]
0,France,2017,332.48,67.00,4.962388
1,France,2018,322.17,67.14,4.798481
2,France,2019,318.95,67.29,4.739932
3,France,2020,316.19,67.43,4.689159
4,France,2021,300.26,67.57,4.443688
5,France,2022,293.12,67.71,4.329050
6,France,2023,295.04,67.86,4.347775
7,France,2024,283.84,68.00,4.174118
8,Switzerland,2017,47.65,8.40,5.672619
9,Switzerland,2018,50.57,8.47,5.970484


## Summary Stats

In [ ]:
df.loc[:,["CO2_emissions[MtCO2]","population[milpeople]", "CO2_emissions_per_capita[tCO2/person]"]].describe()

## Make box plots

In [ ]:
# all data
fig = px.box(df, y="CO2_emissions_per_capita[tCO2/person]",
             labels={
                 "O2_emissions_per_capita[tCO2/person]": "CO2 emissions per capita (tCO2 per person)"
                 },
             title="Distribution of CO2 emissions per capita"
             )
fig.show()

In [ ]:
# by country
fig = px.box(df, x="Country", y="CO2_emissions_per_capita[tCO2/person]", color = "Country",
             labels={
                 "O2_emissions_per_capita[tCO2/person]": "CO2 emissions per capita (tCO2 per person)"
                 },
             title="Distribution of CO2 emissions per capita per country"
             )
fig.show()

## Make time series of average CO2 emissions

In [ ]:
# prepare dataframe
df_graph = pd.melt(df, id_vars = ["Country","Year"])
df_graph = df_graph.loc[df_graph["variable"] == 'CO2_emissions[MtCO2]',:]
df_graph = df_graph.groupby(["Year","variable"], as_index=False)['value'].agg("mean")

# make graph
fig = px.line(df_graph,x="Year", y="value", color="variable", 
              labels={"Year": "Year",
                      "value": "CO2 emissions (MtCO2)",
                      "variable": ""
                      },
              title="Average CO2 emissions across countries"
              )
fig.show()

## Make time series of CO2 emissions by country

In [ ]:
# prepare dataframe
df_graph = pd.melt(df, id_vars = ["Country","Year"])
df_graph = df_graph.loc[df_graph["variable"] == 'CO2_emissions[MtCO2]',:]

# make graph
fig = px.line(df_graph,x="Year", y="value", color="Country", 
              labels={"Year": "Year",
                      "value": "CO2 emissions (MtCO2)",
                      "variable": ""
                      },
              title="CO2 emissions by country"
              )
fig.show()

## Make a function to do this for each variable

In [ ]:
def plot_variable(df, variable):
    
    # prepare dataframe
    df_graph = pd.melt(df, id_vars = ["Country","Year"])
    df_graph = df_graph.loc[df_graph["variable"] == variable,:]

    # make graph
    fig = px.line(df_graph,x="Year", y="value", color="Country", 
                  labels={"Year": "Year",
                          "value": "CO2 emissions (MtCO2)",
                          "variable": ""
                          },
                  title= f"{variable} by country"
                  )
    fig.show()

In [ ]:
plot_variable(df, "CO2_emissions[MtCO2]")

In [ ]:
plot_variable(df, 'population[milpeople]')

In [ ]:
plot_variable(df, 'CO2_emissions_per_capita[tCO2/person]')

## Make time series of poulation and emissions with facets

In [ ]:
# prepare dataframe
df_graph = pd.melt(df, id_vars = ["Country","Year"])
df_graph = df_graph.groupby(["Year","variable"], as_index=False)['value'].agg("mean")

# make graph
fig = px.line(df_graph,x="Year", y="value", color="variable", 
              facet_col="variable", # put variables in different panels (facets)
              facet_col_wrap=2, # say that you want 2 columns (so if you have 3 graphs it will be also 2 rows)
              labels={"Year": "Year",
                      "value": "",
                      "variable": "Variable"
                      },
              title="Average variables across countries"
              )
fig.update_yaxes(matches=None, showticklabels=True) # let y axis scale automatically
fig.show()

## Make scatter of emissions and population

In [ ]:
# normal scatter
df_graph = df.copy()
fig = px.scatter(df_graph, x="population[milpeople]", y="CO2_emissions[MtCO2]",
                 color="Country",           # optional: see each country separately
                 trendline="ols",           # adds regression line
                 trendline_scope="overall", # regression over all points (default)
                 labels={
                     "population[milpeople]": "Population (million people)",
                     "CO2_emissions[MtCO2]": "CO2 emissions (MtCO2)"
                 },
                 title="CO2 emissions vs. population"
)
fig.show()

## Make scatter of mean of emissions and population across years

In [ ]:
# prepare dataframe
df_graph = pd.melt(df, id_vars = ["Country","Year"])
df_graph = df_graph.groupby(["Country","variable"], as_index=False)['value'].agg("mean")
df_graph = df_graph.pivot(index="Country", columns="variable", values="value").reset_index()

# make scatter
fig = px.scatter(df_graph, x="population[milpeople]", y="CO2_emissions[MtCO2]",
                 color="Country",           # optional: see each country separately
                 trendline="ols",           # adds regression line
                 trendline_scope="overall", # regression over all points (default)
                 labels={
                     "population[milpeople]": "Population (million people)",
                     "CO2_emissions[MtCO2]": "CO2 emissions (MtCO2)"
                 },
                 title="CO2 emissions vs. population"
)
fig.show()

## Do the same by avoiding melting and pivoting

In [ ]:
# alternate way (avoid melting and pivot)
df_graph = df.groupby("Country", as_index=False).mean()
df_graph.drop(columns=["Year"],inplace=True)
fig = px.scatter(df_graph, x="population[milpeople]", y="CO2_emissions[MtCO2]",
                 color="Country",           # optional: see each country separately
                 trendline="ols",           # adds regression line
                 trendline_scope="overall", # regression over all points (default)
                 labels={
                     "population[milpeople]": "Population (million people)",
                     "CO2_emissions[MtCO2]": "CO2 emissions (MtCO2)"
                 },
                 title="CO2 emissions vs. population"
)
fig.show()